# TabPFN v1 Prior Sanity Check

Small interactive checks for the vendored `tabpfn_prior` package used by the PFNs training configs. More detailed prior visualizations live in [prior_plots.ipynb](prior_plots.ipynb).

## TabPFN v1 prior

The prior implementation is vendored in `prior-repos/tabpfn-v1-prior` and is installed with `pip install -e ./prior-repos/tabpfn-v1-prior` from the repository root.

In [ ]:
import torch
from tabpfn_prior import build_tabpfn_prior

prior = build_tabpfn_prior(
    prior_type="mlp",
    num_steps=1000,  # how many steps until prior is exhausted
    batch_size=1,  # all datasets in one batch have the same feature/datapoint/class counts
    num_datapoints_max=1000,
    num_features=11,
    max_num_classes=5,
    device="cpu",
    flexible=True,
    return_categorical_mask=True,
    differentiable=True,
    nan_handling=True,
)

# Generate synthetic data
for batch in prior:
    x = batch["x"]  # Input features [batch_size, seq_len, num_features]
    y = batch["y"]  # Labels [batch_size, seq_len]
    target_y = batch["target_y"]  # Target labels (same as y)
    eval_pos = batch.get("single_eval_pos")  # Prior metadata from tabpfn-v1-prior
    categorical_mask = batch.get("categorical_mask", None)  # Categorical feature mask

    for i in range(x.shape[0]):
        print(
            f"Sample {i}: x.shape={x[i].shape}, y.shape={y[i].shape}, "
            f"unique_classes={torch.unique(y[i])}, first 5 x: {x[i][:5]}, "
            f"first 5 y: {y[i][:5]}"
        )
        print(f"Number of nans in x: {torch.isnan(x[i]).sum().item()}")
        if torch.isnan(x[i]).sum().item() > 0:
            print(f"Indices of nans in x: {torch.nonzero(torch.isnan(x[i]), as_tuple=True)}")
        if categorical_mask is not None and categorical_mask[i].sum() > 0:
            print(
                "Categorical features at indices: "
                f"{torch.nonzero(categorical_mask[i], as_tuple=True)[0].tolist()}"
            )
    break

print(f"Generated data: x.shape={x.shape}, y.shape={y.shape}")
print(f"target_y.shape={target_y.shape}")
eval_pos_shape_or_value = None if eval_pos is None else getattr(eval_pos, "shape", eval_pos)
print(f"eval_pos shape/value: {eval_pos_shape_or_value}")


In [ ]:
assert x.shape == (1, 1000, 11)
assert y.shape == (1, 1000)
assert target_y.shape == y.shape
assert y.min().item() >= -100
assert y.max().item() < 5

if categorical_mask is not None:
    assert categorical_mask.shape == (1, 11)
    assert categorical_mask.dtype == torch.bool

if eval_pos is not None and hasattr(eval_pos, "shape"):
    assert eval_pos.shape[0] == x.shape[0]

print("TabPFN prior sanity check passed.")
